In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf Diffusion-Illusion # 把我之前写错的那个也删掉
!rm -rf master.zip

# 2. 克隆仓库 (注意：这次名字是对的！)
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    
    # 4. 进入目录
    %cd Diffusion-Illusions
    
    # 5. 安装依赖
    print("正在安装依赖 (红色警告请忽略)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
    
else:
    print("❌❌ 克隆还是失败了，请检查网络。")

In [ ]:
import os

# 定义仓库名字（注意带 's'）
repo_name = "Diffusion-Illusions"

# 检查当前是否已经在文件夹里了
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    # 如果不在，就尝试进去
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        # 如果文件夹都不存在，说明之前的克隆没成功，重新克隆一下
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
from rp import *
import numpy as np
import rp
import torch
import torch.nn as nn
import torch.nn.functional as F
import source.stable_diffusion as sd
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel
from itertools import chain
import torchvision.transforms.functional as TF
import math
from google.colab import files
from PIL import Image, ImageOps

# === A100 专属：高分辨率与高精度采样设置 ===
RESOLUTION = 512  # 提升至 512 解决模糊问题

def create_cylindrical_grid_final(size=RESOLUTION, r_min=0.2, r_max=0.98, device='cuda'):
    """预计算 512 分辨率的高精度采样网格"""
    with torch.no_grad():
        y_range = torch.linspace(-1, 1, size, device=device)
        x_range = torch.linspace(-1, 1, size, device=device)
        grid_y, grid_x = torch.meshgrid(y_range, x_range, indexing='ij')
        
        # 映射逻辑
        theta = grid_x * math.pi + math.pi / 2 
        r = (1 - (grid_y + 1) / 2) * (r_max - r_min) + r_min
        
        # 极坐标转采样点
        sample_x = r * torch.cos(theta)
        sample_y = r * torch.sin(theta)
        
        grid = torch.stack((sample_x, sample_y), dim=2).unsqueeze(0)
    return grid

def create_donut_mask_final(size=RESOLUTION, r_min=0.2, device='cuda'):
    """GPU 原生创建的高精度遮罩"""
    with torch.no_grad():
        y = torch.linspace(-1, 1, size, device=device)
        x = torch.linspace(-1, 1, size, device=device)
        grid_y, grid_x = torch.meshgrid(y, x, indexing='ij')
        dist = torch.sqrt(grid_x**2 + grid_y**2)
        mask = (dist >= r_min).float().unsqueeze(0)
    return mask

def fast_mirror_reflection_sharp(img_tensor, grid):
    """使用 bicubic (双三次插值) 模式，显著提升还原清晰度"""
    return F.grid_sample(img_tensor, grid, mode='bicubic', padding_mode='zeros', align_corners=True)

In [ ]:
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion (A100)...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device

# 预计算并存入显存，确保全程 GPU 运算
MIRROR_GRID = create_cylindrical_grid_final(RESOLUTION, device=device)
DONUT_MASK = create_donut_mask_final(RESOLUTION, device=device)
print(f"✅ 高分辨率网格 ({RESOLUTION}x{RESOLUTION}) 已就绪")

In [ ]:
print(">>> 请上传你的 1:1 谜底图片 (推荐使用 Nanobananapro 生成的人物合影) <<<")
uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))
    target_pil = Image.open(filename).convert('RGB')
    # 强制缩放到 512 以匹配计算分辨率
    target_pil = ImageOps.fit(target_pil, (RESOLUTION, RESOLUTION), method=Image.Resampling.LANCZOS)
    target_tensor = TF.to_tensor(target_pil).to(device)
    # 增加 Batch 维度提升 Loss 计算效率
    target_tensor_4d = target_tensor.unsqueeze(0)
    
    print("\n✅ 目标图片已调整为 512x512 高清模式：")
    rp.display_image(rp.as_numpy_image(target_tensor))

In [ ]:
# === 🎮 核心参数调整：双重语义的平衡 ===
# 之前的 20000 是“独裁模式”，现在降回 3500-4500
# 这个值越高 -> 镜子里越清晰，但地面越像乱码
# 这个值越低 -> 地面越像星云，但镜子里越模糊
GUIDANCE_STRENGTH = 4000 

SD_UPDATE_FREQ = 1       # A100算力充足，我们改为每次都更新SD，保证地面图质量
SD_GUIDANCE_SCALE = 100  # 大幅提高 SD 的权重，强迫地面图必须像Prompt描述的样子

# === 🎨 画面描述 (关键！) ===
# 地面图 Prompt：必须有具体的语义，不能只是“线条”
# 我们选用“漩涡星系”，因为它本身就是圆形的，很适合用来藏圆柱变换的痕迹
prompt_canvas = "A beautiful spiral galaxy, deep space nebula, cosmic dust, blue and purple theme, hubble telescope photography, 8k, detailed, abstract oil painting style"

# 负向提示词：防止生成无关物体
negative_prompt = "faces, text, letters, watermark, humans, buildings, geometric shapes, low quality"

# === 初始化 ===
# 保持高频纹理设置，确保能藏住细节
image_maker = lambda: LearnableImageFourier(
    height=RESOLUTION, 
    width=RESOLUTION, 
    hidden_dim=256, 
    num_features=512, 
    scale=20 
).to(device)

raw_canvas = image_maker()
get_canvas = lambda: raw_canvas() * DONUT_MASK
label_canvas = NegativeLabel(prompt_canvas, negative_prompt)

# 稍微降低学习率，让两个 Loss 能以此拉锯
optim = torch.optim.SGD(raw_canvas.parameters(), lr=1e-4) 

print("✅ 双重语义模式已启动：地面是'星系'，镜中是'你的名字'。")

In [ ]:
NUM_ITER = 4000  # 增加一点迭代次数，因为拔河需要时间收敛
DISPLAY_INTERVAL = 200    

model_sd.max_step = 980
model_sd.min_step = 20
display_eta = rp.eta(NUM_ITER, title='Dual-Semantic Generation')

print(f"🚀 开始生成视错觉... (前 500 步主要在画星系，后面开始藏图)")

try:
    for iter_num in range(NUM_ITER):
        curr_canvas = get_canvas()

        # --- A. 显性语义 (地面图 -> 像星系) ---
        # 这一步保证直接看地面图是有意义的
        _ = model_sd.train_step(
            label_canvas.embedding,
            curr_canvas[None],
            noise_coef=0.1,
            guidance_scale=SD_GUIDANCE_SCALE
        )

        # --- B. 隐性语义 (反射图 -> 像海报) ---
        # 1. 模拟反射
        sim_ref = fast_mirror_reflection_sharp(curr_canvas[None], MIRROR_GRID)
        
        # 2. 动态权重策略 (Warm-up Strategy)
        # 前 500 步让 SD 自由发挥画底色，500步后开始逐渐加强隐写力度
        if iter_num < 500:
            current_strength = 0  # 先别藏
        elif iter_num < 1000:
            # 线性增加权重
            progress = (iter_num - 500) / 500
            current_strength = GUIDANCE_STRENGTH * progress
        else:
            current_strength = GUIDANCE_STRENGTH

        # 3. 计算隐写 Loss
        loss_illusion = F.mse_loss(sim_ref, target_tensor_4d) * current_strength
        
        # 4. 反向传播
        if current_strength > 0:
            loss_illusion.backward()
        
        # 注意：SD 的梯度已经在 train_step 里算好了并累积到了 parameters 中
        # 所以这里不需要再对 loss_sd backward，直接 step 即可
        
        optim.step()
        optim.zero_grad(set_to_none=True)

        # --- C. 监控进度 ---
        if iter_num % DISPLAY_INTERVAL == 0:
            with torch.no_grad():
                canvas_np = rp.as_numpy_image(curr_canvas)
                ref_np = rp.as_numpy_image(sim_ref[0])
                
                from IPython.display import clear_output
                clear_output(wait=True)
                display_eta(iter_num)
                
                print(f"当前阶段: {'画底色(星系)' if iter_num < 500 else '正在注入海报隐写...'}")
                print(f"左侧：肉眼看到的地面图 (应像星系) | 右侧：圆柱镜中像 (应像海报)")
                rp.display_image(np.hstack([canvas_np, ref_np]))

except KeyboardInterrupt:
    print("训练停止。")

In [ ]:
print("==== 最终成果保存 ====")

final_canvas = get_canvas()
final_reflection = fast_mirror_reflection_sharp(final_canvas[None], MIRROR_GRID).squeeze(0)

# 保存地面图 (用于打印)
img_save = TF.to_pil_image(final_canvas.cpu().clamp(0, 1))
img_save.save("your_name_anamorphosis_512.png")

print("1. 请下载 your_name_anamorphosis_512.png 并打印。")
rp.display_image(rp.as_numpy_image(final_canvas))

print("\n2. 镜面反射预览：")
rp.display_image(rp.as_numpy_image(final_reflection))